# Anatomy of a short squeeze: GameStop, 2021

**pyportfolios.com case study CS15** · GME, XRT, ^VIX daily, Oct 2020 – Mar 2021 · NumPy · pandas · matplotlib · yfinance

Real data, real event study. We reconstruct the January 2021 squeeze from the tape:
the run-up, the exact +134.8% and −60% days, the spillover into XRT (the ETF whose
short interest made it a squeeze proxy) and the VIX, the mark-to-market P&L of a
short held through the event, and the pre-event VaR that saw none of it coming.

*Prices are split-adjusted (GME split 4-for-1 in July 2022): the famous $347.51
close on Jan 27 appears as $86.88. Returns are unaffected.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.stats import norm

plt.rcParams["figure.figsize"] = (10, 5)

## 1 · Data: GME, XRT and the VIX

Three series tell the story: the stock, the sector ETF that the short interest was
partly routed through, and the market's fear gauge.

In [ ]:
px = yf.download(["GME", "XRT", "^VIX"], start="2020-10-01", end="2021-04-01",
                 auto_adjust=True, progress=False)["Close"].dropna()
gme, xrt, vix = px["GME"], px["XRT"], px["^VIX"]
gme.plot(title="GME close (split-adjusted), Oct 2020 – Mar 2021");

## 2 · Run-up, realised vol, short P&L

Anchor a $1 short at the 2020 year-end close and mark it daily. A short's loss is
unbounded — this series shows exactly how unbounded it got.

In [ ]:
ret = gme.pct_change().dropna()
logret = np.log(gme).diff().dropna()
anchor = float(gme.loc["2020-12-31"])

run_up = gme.max() / anchor - 1
rvol = logret.rolling(10).std() * np.sqrt(252)
short_pnl = 1 - gme.loc["2020-12-31":] / anchor      # negative = loss

print(f"year-end close   ${anchor:.2f}  (split-adjusted)")
print(f"peak close       ${gme.max():.2f}  on {gme.idxmax().date()}")
print(f"run-up           {run_up:+.0%}")
print(f"peak 10d rvol    {rvol.max():.0%} annualised")
print(f"worst short P&L  {short_pnl.min():+.0%} of the initial proceeds")
short_pnl.plot(title="P&L of a $1 short opened 2020-12-31, marked daily");

## 3 · The event tape, day by day

The two bars every risk report should have flagged: **+134.8% on Jan 27** and
**−60.0% on Feb 2**.

In [ ]:
events = ["2021-01-13", "2021-01-22", "2021-01-25", "2021-01-26",
          "2021-01-27", "2021-01-28", "2021-02-01", "2021-02-02"]
tape = pd.DataFrame({"close": gme.loc[events].round(2),
                     "return": (ret.loc[events] * 100).round(1)})
print(tape)

win = ret.loc["2021-01-19":"2021-02-04"] * 100
win.index = win.index.strftime("%m/%d")              # categorical axis for the bars
win.plot.bar(title="GME daily returns, Jan 19 - Feb 4 (%)")
plt.axhline(0, color="k", lw=0.5);

## 4 · Spillover: XRT and the VIX

XRT held GME, and short interest routed through the ETF meant the squeeze dragged
the whole wrapper with it; the VIX repriced the same week.

In [ ]:
idx = 100 * px / px.iloc[0]
idx[["XRT", "^VIX"]].plot(title="XRT and ^VIX, rebased to 100 at Oct 1 2020")

sq = slice("2021-01-11", "2021-02-05")            # the squeeze weeks
xs, vs = xrt.loc[sq], vix.loc[sq]
print(f"XRT squeeze peak  {xs.max():.2f} on {xs.idxmax().date()}"
      f"  ({xs.max()/xrt.loc['2020-12-31']-1:+.1%} vs year-end)")
print(f"VIX squeeze peak  {vs.max():.2f} on {vs.idxmax().date()}")

## 5 · Dealer gamma: hedge demand vs price

Black–Scholes delta of the calls retail was buying: as spot runs toward and through
the strike, the dealer's hedge ratio sprints from a few shares per contract to ~100.

In [ ]:
def call_delta(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d1)

for S in [40, 60, 80, 100]:
    print(S, round(100 * call_delta(S, K=60, T=0.05, r=0.0, sigma=1.2), 1),
          "shares/contract")

## 6 · The VaR that was blind

Calibrate a historical 99% one-day VaR on GameStop's own Oct–Dec 2020 returns —
the standard calm-sample setup — then compare it with what Jan 27 delivered.

In [ ]:
pre = ret.loc[:"2020-12-31"]
var99 = np.percentile(pre, 1)
jan27 = float(ret.loc["2021-01-27"])

print(f"pre-event daily sigma   {pre.std(ddof=1):.2%}")
print(f"historical 99% 1d VaR   {var99:.2%}")
print(f"Jan 27 actual return    {jan27:+.1%}")
print(f"... i.e. {jan27 / pre.std(ddof=1):.0f} sigma on the calm calibration")

## Takeaways

- Short interest above float turns sellers into *contractually forced* buyers; price
  is the only free variable.
- The gamma loop is the accelerant: dealer hedge demand ramps with spot.
- The blow-up was invisible to price-history VaR — the leading indicators were
  positioning numbers (SI/float, days-to-cover, options OI), not returns.
- A $1 short held from year-end lost a multiple of its proceeds; sizing for the
  tail is the only defence.

*© pyportfolios.com — runnable companion to the case study. Data: Yahoo Finance via yfinance.*